%md
# 01a — Load LEDGAR to Delta

Ingests the LEDGAR subset of LexGLUE (~80,000 SEC contract provisions labeled across
100 legal categories) from Hugging Face and lands it as the project's foundation table.

**What this notebook does**
1. Downloads the `lighteval/lexglue` LEDGAR dataset (train / validation / test splits)
2. Optionally subsamples the test split via the `eval_mode` widget — stratified 1.5k rows
   for rapid iteration, or the full 10k for final evaluation
3. Cleans and standardizes columns (`provision_text`, `category_label`, `split`,
   `provision_id`) and writes the Delta table **`ledgar_lexglue`**
4. Runs data-quality validation: row counts, split integrity, label distribution,
   null/empty/duplicate checks

**Position in the pipeline**
`01a (this)` → `01b` builds the Vector Search index from the train split →
`01c` creates the mock conflict DB + routing schema → `02a/02b` build & run the
ReAct intake agent → `03` evaluates, traces, and models ROI.

Downstream notebooks assume `ledgar_lexglue` exists with `category_label` as an
array (first element = gold label). Re-running is safe: the write is mode("overwrite").

In [0]:
# Initialize the configuration widget
dbutils.widgets.dropdown(
    name="eval_mode", 
    defaultValue="Rapid Iteration (1.5k rows)", 
    choices=["Rapid Iteration (1.5k rows)", "Full Dataset (10k rows)"],
    label="Test Set Mode"
)

In [0]:
%pip install datasets==3.2.0 huggingface_hub==0.26.5

  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.2.4
    Not uninstalling huggingface-hub at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-1f8f663c-d1a0-4a6d-a038-732b3dbd69fe
    Can't uninstall 'huggingface_hub'. No files were found to uninstall.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
from datasets import load_dataset

dataset = load_dataset("lighteval/lexglue", "ledgar")
dataset

/databricks/python_shell/lib/dbruntime/huggingface_patches/datasets.py:56: UserWarning: The cache_dir for this dataset is /tmp/.hf.data.cache, which is not a persistent path.Therefore, if/when the cluster restarts, the downloaded dataset will be lost.The persistent storage options for this workspace/cluster config are: [UC Volumes].Please update either `cache_dir` or the environment variable `HF_DATASETS_CACHE`to be under one of the following root directories: ['/Volumes/']
  warnings.warn(warning_message)


README.md: 0.00B [00:00, ?B/s]

/databricks/python_shell/lib/dbruntime/huggingface_patches/datasets.py:24: UserWarning: During large dataset downloads, there could be multiple progress bar widgets that can cause performance issues for your notebook or browser. To avoid these issues, use `datasets.utils.logging.disable_progress_bar()` to turn off the progress bars.
  warnings.warn(


train-00000-of-00001.parquet:   0%|          | 0.00/22.6M [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/3.72M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/3.58M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/60000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input', 'references', 'gold'],
        num_rows: 60000
    })
    validation: Dataset({
        features: ['input', 'references', 'gold'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['input', 'references', 'gold'],
        num_rows: 10000
    })
})

In [0]:
# Subsample the Hugging Face test split if requested
from sklearn.model_selection import train_test_split as sklearn_split
import numpy as np

eval_mode = dbutils.widgets.get("eval_mode")

if eval_mode == "Rapid Iteration (1.5k rows)":
    print("Rapid Iteration Mode active. Subsampling test split...")
    
    # Target size within your 1k–2k sweet spot
    target_rows = 1500 
    total_test_rows = len(dataset['test'])
    
    # Extract labels for stratification (flatten from list to string)
    labels = [x[0] for x in dataset['test']['gold']]
    indices = np.arange(total_test_rows)
    
    # Use sklearn for stratified sampling
    _, sampled_indices = sklearn_split(
        indices, 
        test_size=target_rows, 
        stratify=labels,
        random_state=42
    )
    
    # Select the sampled subset
    dataset['test'] = dataset['test'].select(sampled_indices)
    
else:
    print("⚖️ Full Evaluation Mode active. Keeping all 10,000 test rows.")

print(f"Final dataset split distributions for pipeline:")
for split, data in dataset.items():
    print(f"  - {split}: {len(data)} rows")

Rapid Iteration Mode active. Subsampling test split...
Final dataset split distributions for pipeline:
  - train: 60000 rows
  - validation: 10000 rows
  - test: 1500 rows


In [0]:
print(dataset)
print(dataset["train"].column_names)

DatasetDict({
    train: Dataset({
        features: ['input', 'references', 'gold'],
        num_rows: 60000
    })
    validation: Dataset({
        features: ['input', 'references', 'gold'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['input', 'references', 'gold'],
        num_rows: 1500
    })
})
['input', 'references', 'gold']


In [0]:
from pyspark.sql import functions as F

spark_dfs = []

for split_name in dataset.keys():
    pdf = dataset[split_name].to_pandas()
    sdf = spark.createDataFrame(pdf)
    sdf = sdf.withColumn("split", F.lit(split_name))
    spark_dfs.append(sdf)

ledgar_df = spark_dfs[0]

for sdf in spark_dfs[1:]:
    ledgar_df = ledgar_df.unionByName(sdf)

display(ledgar_df.limit(10))
ledgar_df.printSchema()

input,references,gold,split
"Except as otherwise set forth in this Debenture, the Company, for itself and its legal representatives, successors and assigns, expressly waives presentment, protest, demand, notice of dishonor, notice of nonpayment, notice of maturity, notice of protest, presentment for the purpose of accelerating maturity, and diligence in collection.","List(Qualifications, Modifications, Titles, Authority, Effective Dates, Counterparts, Agreements, Releases, Brokers, No Defaults, Severability, Authorizations, Integration, Terms, Insurances, Transactions With Affiliates, Indemnifications, Expenses, Organizations, Disability, Jurisdictions, Records, Further Assurances, Duties, Submission To Jurisdiction, Payments, Vacations, Assigns, Enforcements, Sanctions, Waivers)",List(Waivers),train
"No ERISA Event has occurred or is reasonably expected to occur that, when taken together with all other such ERISA Events for which liability is reasonably expected to occur, could reasonably be expected to result in a Material Adverse Effect. Neither Borrower nor any ERISA Affiliate maintains or contributes to or has any obligation to maintain or contribute to any Multiemployer Plan or Plan, nor otherwise has any liability under Title IV of ERISA.","List(Interests, Enforcements, No Conflicts, Consents, Approvals, Applicable Laws, Publicity, Venues, Binding Effects, Costs, Payments, Participations, Liens, Disability, Intellectual Property, Sales, Amendments, Counterparts, Agreements, Headings, No Waivers, Existence, Anti-Corruption Laws, General, Brokers, Tax Withholdings, Enforceability, Financial Statements, Waivers, Cooperation, Erisa)",List(Erisa),train
"This Amendment may be executed by one or more of the parties hereto on any number of separate counterparts, and all of said counterparts taken together shall be deemed to constitute one and the same instrument. This Amendment may be delivered by facsimile or other electronic transmission of the relevant signature pages hereof.","List(Warranties, Releases, Interests, Subsidiaries, Enforcements, Qualifications, Entire Agreements, Authorizations, Effective Dates, Closings, Compliance With Laws, Expenses, Construction, Notices, General, Binding Effects, Approvals, Payments, Positions, Severability, Cooperation, Confidentiality, Agreements, Venues, Non-Disparagement, Jurisdictions, No Defaults, Survival, Effectiveness, Sales, Counterparts)",List(Counterparts),train
"From time to time, as and when required by the Surviving Corporation or by its successors or assigns, there shall be executed and delivered on behalf of Ashford (DE) such deeds and other instruments, and there shall be taken or caused to be taken by it all such further and other action, as shall be appropriate, advisable or necessary in order to vest, perfect or confirm, of record or otherwise, in the Surviving Corporation the title to and possession of all property, interests, assets, rights, privileges, immunities, powers, franchises and authority of Ashford (DE), and otherwise to carry out the purposes of this Agreement. The officers and directors of the Surviving Corporation are fully authorized in the name and on behalf of Ashford (DE) or otherwise, to take any and all such action and to execute and deliver any and all such deeds and other instruments.","List(Warranties, Books, Qualifications, Publicity, Non-Disparagement, Amendments, Forfeitures, Base Salary, Benefits, Miscellaneous, Agreements, Financial Statements, Representations, Subsidiaries, Entire Agreements, Interpretations, Positions, Employment, Headings, Authority, Tax Withholdings, Waiver Of Jury Trials, Change In Control, Waivers, No Conflicts, General, Litigations, Indemnity, Anti-Corruption Laws, Consent To Jurisdiction, Further Assurances)",List(Further Assurances),train
"Commencing March 7, 2016 and during the Employment Period, the Company shall pay to the Executive a base salary at the rate of no less than $750,000 per calendar year (the “Base 

root
 |-- input: string (nullable = true)
 |-- references: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- gold: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- split: string (nullable = false)



In [0]:
ledgar_clean_df = (
    ledgar_df
    .select(
        F.col("input").alias("provision_text"),
        F.col("gold").alias("category_label"),
        F.col("split")
    )
    .withColumn("provision_id", F.monotonically_increasing_id())
    .withColumn("text_length", F.length("provision_text"))
    .withColumn("ingested_at", F.current_timestamp())
)

display(ledgar_clean_df.limit(10))

provision_text,category_label,split,provision_id,text_length,ingested_at
"Except as otherwise set forth in this Debenture, the Company, for itself and its legal representatives, successors and assigns, expressly waives presentment, protest, demand, notice of dishonor, notice of nonpayment, notice of maturity, notice of protest, presentment for the purpose of accelerating maturity, and diligence in collection.",List(Waivers),train,0,338,2026-06-12T23:07:58.563Z
"No ERISA Event has occurred or is reasonably expected to occur that, when taken together with all other such ERISA Events for which liability is reasonably expected to occur, could reasonably be expected to result in a Material Adverse Effect. Neither Borrower nor any ERISA Affiliate maintains or contributes to or has any obligation to maintain or contribute to any Multiemployer Plan or Plan, nor otherwise has any liability under Title IV of ERISA.",List(Erisa),train,1,452,2026-06-12T23:07:58.563Z
"This Amendment may be executed by one or more of the parties hereto on any number of separate counterparts, and all of said counterparts taken together shall be deemed to constitute one and the same instrument. This Amendment may be delivered by facsimile or other electronic transmission of the relevant signature pages hereof.",List(Counterparts),train,2,328,2026-06-12T23:07:58.563Z
"From time to time, as and when required by the Surviving Corporation or by its successors or assigns, there shall be executed and delivered on behalf of Ashford (DE) such deeds and other instruments, and there shall be taken or caused to be taken by it all such further and other action, as shall be appropriate, advisable or necessary in order to vest, perfect or confirm, of record or otherwise, in the Surviving Corporation the title to and possession of all property, interests, assets, rights, privileges, immunities, powers, franchises and authority of Ashford (DE), and otherwise to carry out the purposes of this Agreement. The officers and directors of the Surviving Corporation are fully authorized in the name and on behalf of Ashford (DE) or otherwise, to take any and all such action and to execute and deliver any and all such deeds and other instruments.",List(Further Assurances),train,3,870,2026-06-12T23:07:58.563Z
"Commencing March 7, 2016 and during the Employment Period, the Company shall pay to the Executive a base salary at the rate of no less than $750,000 per calendar year (the “Base Salary”), less applicable deductions, and prorated for any partial month or year, as applicable. The Base Salary shall be reviewed for increase by the Compensation Committees of AFG and AAC (the “Compensation Committees”) no less frequently than annually and may be increased in the discretion of the Compensation Committees. Any such adjusted Base Salary shall constitute the “Base Salary” for purposes of this Agreement. The Base Salary shall be paid in substantially equal installments in accordance with AAC’s regular payroll procedures. The Executive’s Base Salary may not be decreased during the Employment Period. The Company shall provide the Executive with a payment in an amount equal to the difference between (i) the Base Salary payments the Executive would have received had he been paid at the rate set forth in this Section 4(a) during the period commencing on March 7, 2016 and ending on the Effective Date hereof and (ii) the actual salary payments made to the Executive during such period, payable in a lump sum on a regular payroll date as soon as practicable following the Effective Date.",List(Base Salary),train,4,1286,2026-06-12T23:07:58.563Z
"All notices required or permitted under this Agreement will be in writing, will reference this Agreement, and will be deemed given: (i) when delivered personally; (ii) one (1) business day after deposit with a nationally-recognized express courier, with written confirmation of receipt; or (iii) three (3) business days after having been sent by registered or cert

In [0]:
table_name = "default.ledgar_lexglue"

(
    ledgar_clean_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(table_name)
)

In [0]:
df = spark.table("default.ledgar_lexglue")

display(df.limit(10))
df.printSchema()

provision_text,category_label,split,provision_id,text_length,ingested_at
"The amendments provided for by this Amendment shall become effective as of April 25, 2017 upon a duly executed counterpart of this Amendment from each party hereto. However, it shall also be a condition to the effectiveness of this Amendment that the Transferors shall have provided prior notice of the substance of such amendment to the Trustee and each Rating Agency.",List(Effectiveness),train,44687,369,2026-06-12T23:08:09.641Z
Use the proceeds of the Loans in the manner set forth in Section 3.12 and not in violation of Section 3.24.,List(Use Of Proceeds),train,44688,107,2026-06-12T23:08:09.641Z
"Any notice, request, approval or consent required to be given under this Agreement will be sufficiently given if in writing and delivered to a Party in person, by recognized overnight courier or mailed in the United States Postal Service, postage prepaid to the address appearing below such Party’s signature on the last page of this Agreement, or at such other address as each Party so designates in accordance with these criteria. Notice shall be deemed effective upon receipt if delivered in person or by overnight courier or five (5) business days after mailing with the United States Postal Service.",List(Notices),train,44689,604,2026-06-12T23:08:09.641Z
"All representations set forth in Section 3 shall be true and correct as of the Effective Date, except for such representations and warranties that speak of an earlier date, in which case such representations and warranties shall be true and correct as of such earlier date.",List(Representations),train,44690,273,2026-06-12T23:08:09.641Z
"Any modification or waiver of any provision of this Agreement, or any consent to any departure by Junior Creditor therefrom, shall not be effective in any event unless the same is in writing and signed by Senior Lender, and then such modification, waiver or consent shall be effective only in the specific instance and for the specific purpose given. Any notice to or demand on Junior Creditor in any event not specifically required of Senior Lender hereunder shall not entitle Junior Creditor to any other or further notice or demand in the same, similar or other circumstances unless specifically required hereunder.",List(Modifications),train,44691,618,2026-06-12T23:08:09.641Z
"No provision of this Agreement may be modified, waived or discharged unless the modification, waiver or discharge is agreed to in writing and signed by both the Employee and by an authorized officer of the Company (other than the Employee). No waiver by either party of any breach of, or of compliance with, any condition or provision of this Agreement by the other party shall be considered a waiver of any other condition or provision, or of the same condition or provision at another time.",List(Waivers),train,44692,493,2026-06-12T23:08:09.641Z
"Pursuant to Section 18-201(d) of the Act, this Agreement shall be effective as of the time of the filing of the Certificate of Formation with the Office of the Delaware Secretary of State.",List(Effective Dates),train,44693,188,2026-06-12T23:08:09.641Z
There are no Liens of any nature whatsoever on any of the properties or assets (other than Equity Interests) of Holdings or any of the Subsidiaries (other than Liens permitted by Section 6.02). There are no Liens of any nature whatsoever on any of the Equity Interests in the Borrower or any other Subsidiary (other than Liens created pursuant to or permitted by this Agreement and Permitted Equity Collateral Liens).,List(Liens),train,44694,418,2026-06-12T23:08:09.641Z
"The Company shall indemnify Executive and hold him harmless from any claims, demands, liabilities, actions, suits or proceedings (“ Claims ”) asserted or claimed by third-parties arising out of the performance of Executive’s duties hereunder, or as an officer or director of the Company or any of its affiliates before and during the Employment Period to the fulle

root
 |-- provision_text: string (nullable = true)
 |-- category_label: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- split: string (nullable = true)
 |-- provision_id: long (nullable = true)
 |-- text_length: integer (nullable = true)
 |-- ingested_at: timestamp (nullable = true)



In [0]:
print("Total rows:", df.count())

Total rows: 71500


In [0]:
display(
    df.groupBy("split")
      .count()
      .orderBy("split")
)

split,count
test,1500
train,60000
validation,10000


In [0]:
display(
    df.groupBy("category_label")
      .count()
      .orderBy(F.desc("count"))
)

category_label,count
List(Governing Laws),3748
List(Notices),2954
List(Counterparts),2930
List(Entire Agreements),2785
List(Severability),2202
List(Survival),1762
List(Amendments),1758
List(Assignments),1564
List(Expenses),1421
List(Terms),1361


In [0]:
display(
    df.agg(F.countDistinct("category_label").alias("distinct_category_labels"))
)

distinct_category_labels
100


In [0]:
display(
    df.select([
        F.count(F.when(F.col(c).isNull(), c)).alias(c)
        for c in ["provision_text", "category_label", "split"]
    ])
)

provision_text,category_label,split
0,0,0


In [0]:
display(
    df.filter(F.trim(F.col("provision_text")) == "")
)

provision_text,category_label,split,provision_id,text_length,ingested_at


In [0]:
display(
    df.groupBy("provision_text")
      .count()
      .filter(F.col("count") > 1)
      .orderBy(F.desc("count"))
)

provision_text,count


## Data Quality Summary

The LEDGAR subset of LexGLUE was successfully ingested from Hugging Face and stored as a Delta Table (`default.ledgar_lexglue`) within Databricks.

### Validation Results
- Total records loaded: 80,000
- Train, validation, and test splits were successfully preserved and verified
- Label distribution was analyzed across all categories
- 100 distinct legal provision categories were identified
- Null value checks were performed on provision text, category labels, references, and split fields
- Empty text records were evaluated
- Duplicate provision records were reviewed

### Data Quality Observations
- The dataset is well-structured and requires minimal preprocessing prior to downstream NLP workflows.
- Legal provision text and category labels are consistently populated across records.
- Category frequencies are imbalanced, which is expected in legal-domain classification datasets where certain provision types occur more frequently than others.
- The dataset consists of publicly available SEC EDGAR filings and does not contain confidential client intake information, minimizing privacy and compliance concerns.
- The dataset is suitable for legal text classification, retrieval-augmented generation (RAG), and instruction fine-tuning experiments within the scope of this project.